# YOLO + UGOT Project: Find the Toy Tiger and Approach It

**Target learners:** Upper primary / lower secondary Python learners  
**Suggested duration:** 10 sessions × about 2 hours  
**Final project:** UGOT uses its camera and a custom YOLO model to recognise a **toy tiger**, align itself with the tiger, move closer, and stop at a safe distance.

---

## Big Mental Model: Sense → Think → Act

### SENSE
UGOT camera captures an image.

### THINK
YOLO answers:
1. **What is it?** → `tiger`
2. **How sure am I?** → confidence
3. **Where is it?** → bounding box / centre position

Python then decides:
- Tiger on the left → move left
- Tiger on the right → move right
- Tiger in the centre → move forward
- Tiger close enough → stop

### ACT
UGOT moves using mecanum-wheel commands.

> **Important teaching idea:** YOLO does not directly drive the robot.  
> YOLO provides information. **Python makes the decision.**

## Course Map

| Session | One Main Idea | Student Deliverable |
|---|---|---|
| 1 | Let YOLO see | Detect objects in a still image |
| 2 | Read YOLO's answers | Print class name and confidence |
| 3 | Find where the object is | Calculate bounding-box centre |
| 4 | Turn position into decisions | Print LEFT / RIGHT / FORWARD |
| 5 | Use live camera frames | Run YOLO repeatedly on camera images |
| 6 | Build a tiger dataset | Collect and label toy-tiger images |
| 7 | Train a custom YOLO model | Produce `best.pt` |
| 8 | Test whether the AI is reliable | Complete a simple test table |
| 9 | Connect YOLO to UGOT | UGOT aligns with the tiger |
| 10 | Final autonomous project | Find → align → approach → stop |

# Before Session 1 — Setup

The original uploaded YOLO notebook already used the Ultralytics workflow:

```python
YOLO("yolo26n.pt")
model.predict(...)
result.boxes
box.cls
box.conf
model.names
```

We will keep the same workflow and gradually add only one major idea at a time.

Run the installation cell once if Ultralytics is not installed.

In [ ]:
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import time
from pathlib import Path
from IPython.display import display, Image

print("Libraries imported successfully.")

---
# Session 1 — Let YOLO See

## Objective
Understand that YOLO can look at an image and return detected objects.

## New core idea
**Image → YOLO → detections**

For now, do not worry about neural-network mathematics. Treat YOLO as a visual detector that gives us useful information.

In [ ]:
# Load a small pretrained YOLO model.
# This is the same starting idea as the uploaded Yolo.ipynb notebook.

model = YOLO("yolo26n.pt")

print("YOLO model loaded.")

### Try YOLO on one image

Put an image file in the same folder as this notebook.

For example:

```text
street.jpg
animals.jpg
classroom.jpg
```

Change the filename below.

In [ ]:
image_path = "animals.jpg"   # <-- Change this filename

results = model.predict(
    image_path,
    conf=0.5,
    verbose=False
)

result = results[0]
annotated = result.plot()

ok, jpg = cv2.imencode(".jpg", annotated)

if ok:
    display(Image(data=jpg.tobytes()))

## What happened?

YOLO did three important jobs:

1. It looked at the picture.
2. It tried to recognise known object classes.
3. It drew bounding boxes around accepted detections.

### Student task
Test YOLO with **three different images**.

Record:
- What objects were detected?
- What objects were missed?
- Was anything detected incorrectly?

---
# Session 2 — What Did YOLO Find?

## Objective
Read the information stored inside each YOLO detection.

## New core idea

A YOLO detection contains:

```text
WHAT?       → class
HOW SURE?   → confidence
WHERE?      → bounding box
```

In this session, we focus only on **WHAT** and **HOW SURE**.

In [ ]:
# Use the most recent result from Session 1.

for box in result.boxes:
    class_id = int(box.cls[0])
    class_name = model.names[class_id]
    confidence = float(box.conf[0])

    print(f"{class_name}: {confidence:.2f}")

### Confidence

A confidence such as:

```text
0.91
```

means YOLO is more confident than:

```text
0.42
```

But remember:

> Increasing the `conf` setting does not make YOLO smarter.  
> It only changes which detections we accept.

In [ ]:
# Experiment with different confidence thresholds.

for threshold in [0.3, 0.5, 0.8]:
    results = model.predict(
        image_path,
        conf=threshold,
        verbose=False
    )

    count = len(results[0].boxes)
    print(f"conf={threshold} -> {count} accepted detections")

### Student challenge

Find the class ID of an object by name.

In [ ]:
item = "dog"

matches = [
    class_id
    for class_id, name in model.names.items()
    if name == item
]

if matches:
    print(f"{item} class ID = {matches[0]}")
else:
    print(f"{item} is not in this pretrained model.")

> **Important for our project:** A standard pretrained model may recognise many animal categories, but our final goal is a specific **toy tiger**.  
> We will therefore create our own one-class dataset and train a custom model later.

---
# Session 3 — Where Is the Object?

## Objective
Use YOLO's bounding-box coordinates to calculate the centre of a detected object.

## New core idea
A bounding box tells us **where** the object is.

YOLO can give coordinates in this form:

```text
(x1, y1) ---------
   |             |
   |   object    |
   |             |
   --------- (x2, y2)
```

The horizontal centre is:

```python
center_x = (x1 + x2) / 2
```

In [ ]:
# Print bounding-box information.

for box in result.boxes:
    class_id = int(box.cls[0])
    class_name = model.names[class_id]

    x1, y1, x2, y2 = box.xyxy[0]
    x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

    center_x = (x1 + x2) // 2
    center_y = (y1 + y2) // 2

    print(
        f"{class_name}: "
        f"box=({x1},{y1}) to ({x2},{y2}), "
        f"center=({center_x},{center_y})"
    )

## Turn coordinates into a simple position

We divide the camera image into three zones:

```text
|-------------|-------------|-------------|
|    LEFT     |   CENTER    |    RIGHT    |
|-------------|-------------|-------------|
```

In [ ]:
# Example only:
frame_width = 640

left_boundary = int(frame_width * 0.40)
right_boundary = int(frame_width * 0.60)

center_x = 200  # Try 200, 320, 500

if center_x < left_boundary:
    position = "LEFT"
elif center_x > right_boundary:
    position = "RIGHT"
else:
    position = "CENTER"

print(position)

### Student task

Change only `center_x`.

Predict the answer before running the cell:

- `center_x = 100`
- `center_x = 320`
- `center_x = 550`

This connects image coordinates to robot navigation.

---
# Session 4 — Turn Detection into Decisions

## Objective
Combine class, confidence, and object position with `if / elif / else`.

## New core idea
**AI information becomes useful only after our program makes a decision.**

For now, the robot does **not move**. We only print the action.

In [ ]:
def decide_action(center_x, frame_width, confidence):
    left_boundary = int(frame_width * 0.40)
    right_boundary = int(frame_width * 0.60)

    if confidence < 0.60:
        return "STOP - not confident enough"

    if center_x < left_boundary:
        return "MOVE LEFT"

    if center_x > right_boundary:
        return "MOVE RIGHT"

    return "MOVE FORWARD"


# Test the thinking function.
print(decide_action(120, 640, 0.90))
print(decide_action(320, 640, 0.90))
print(decide_action(520, 640, 0.90))
print(decide_action(320, 640, 0.30))

## Why make a function?

The function separates:

```text
SENSE        THINK          ACT
camera  →  decision  →  robot movement
```

This makes the program easier to test and debug.

Before controlling real motors, we should prove that the **THINK** stage works correctly.

---
# Session 5 — YOLO with a Live MacBook Camera

## Objective
Run YOLO repeatedly on camera frames.

## New core idea
A video is a sequence of pictures.

```text
frame 1 → YOLO
frame 2 → YOLO
frame 3 → YOLO
...
```

We use a loop to keep getting new frames.

In [ ]:
# MacBook camera example.
# Press the stop button in Jupyter if you want to stop early.

camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)
time.sleep(1)

if not camera.isOpened():
    print("The camera could not be opened.")

else:
    print("Camera is ready.")

    display_area = None
    start_time = time.time()

    while time.time() - start_time < 10:

        frame_ok, frame = camera.read()

        if not frame_ok:
            continue

        results = model.predict(
            frame,
            conf=0.5,
            verbose=False
        )

        annotated = results[0].plot()

        jpg_ok, jpg = cv2.imencode(".jpg", annotated)

        if jpg_ok:
            image = Image(data=jpg.tobytes())

            if display_area is None:
                display_area = display(image, display_id=True)
            else:
                display_area.update(image)

        time.sleep(0.05)

    camera.release()
    print("Camera closed.")

### Student investigation

Try moving objects:
- closer / farther
- left / right
- partly hidden
- in bright / dark conditions

Ask:

> Does YOLO always give the same confidence?

---
# Session 6 — Build a Toy-Tiger Dataset

## Objective
Collect varied images that can teach YOLO what our toy tiger looks like.

## New core idea
**Good AI depends on good examples.**

For a one-class project, our class will simply be:

```text
tiger
```

### Recommended first classroom dataset

Aim for approximately **60–100 images** of the toy tiger.

Include variation:
- front / back / side
- near / far
- high / low camera angle
- bright / dim lighting
- different backgrounds
- partial obstruction
- tiger near other objects

Do not collect 100 nearly identical pictures.

## Optional: collect tiger images with the MacBook camera

This program saves one image every 0.5 seconds.

Create this folder beside the notebook:

```text
raw_tiger_images/
```

In [ ]:
import os

folder = "raw_tiger_images"
os.makedirs(folder, exist_ok=True)

camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)
time.sleep(1)

if not camera.isOpened():
    print("The camera could not be opened.")

else:
    collected = 0
    next_save_time = time.time()
    start_time = time.time()

    print("Move the toy tiger around.")
    print("Collection runs for 20 seconds.")

    while time.time() - start_time < 20:

        frame_ok, frame = camera.read()

        if not frame_ok:
            continue

        if time.time() >= next_save_time:
            filename = os.path.join(
                folder,
                f"tiger_{collected:03d}.jpg"
            )

            cv2.imwrite(filename, frame)
            print("Saved:", filename)

            collected += 1
            next_save_time = time.time() + 0.5

    camera.release()
    print(f"Finished. {collected} images collected.")

## Label the tiger

YOLO detection training needs a bounding box around the tiger in each image.

A YOLO label line looks like:

```text
class_id  x_center  y_center  width  height
```

For our one-class project:

```text
tiger = class 0
```

Example:

```text
0 0.51 0.48 0.34 0.42
```

The coordinates are normalized from `0.0` to `1.0`.

### Recommended folder structure after annotation

```text
tiger_dataset/
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

A simple split is:
- about 80% → training
- about 20% → validation

You may use a YOLO-compatible annotation tool to draw the boxes.  
The important learning goal is **not the annotation software**; it is understanding that each training image needs the correct tiger box.

## Create `data.yaml`

The training configuration tells YOLO:
- where the dataset is
- how many classes exist
- the class name

In [ ]:
from pathlib import Path

dataset_path = Path("tiger_dataset").resolve()

yaml_text = f'''
path: {dataset_path}
train: images/train
val: images/val

names:
  0: tiger
'''.strip()

Path("tiger_data.yaml").write_text(yaml_text)

print(yaml_text)

---
# Session 7 — Train Our Own Toy-Tiger YOLO

## Objective
Use transfer learning to create a model that recognises our toy tiger.

## New core idea

```text
pretrained YOLO
      +
our tiger examples
      ↓
custom tiger detector
```

We do **not** train the whole idea of vision from zero.  
We start from a pretrained model and fine-tune it using our own labelled images.

In [ ]:
# IMPORTANT:
# Run this only after your dataset has been labelled
# and tiger_data.yaml points to the correct folders.

tiger_model = YOLO("yolo26n.pt")

training_results = tiger_model.train(
    data="tiger_data.yaml",
    epochs=30,
    imgsz=640,
    batch=8,
    project="tiger_training",
    name="toy_tiger"
)

## Where is the trained model?

After training, look for a path similar to:

```text
tiger_training/
└── toy_tiger/
    └── weights/
        ├── best.pt
        └── last.pt
```

We normally use:

```text
best.pt
```

Think of `best.pt` as:

> The saved version of the model that performed best during training.

In [ ]:
# Load the custom model after training.

custom_model_path = "tiger_training/toy_tiger/weights/best.pt"

tiger_model = YOLO(custom_model_path)

print("Custom tiger model loaded.")
print(tiger_model.names)

## Test the custom model on a new image

Use an image that was **not** used for training if possible.

In [ ]:
test_image = "test_tiger.jpg"   # <-- Change this

results = tiger_model.predict(
    test_image,
    conf=0.5,
    verbose=False
)

annotated = results[0].plot()

ok, jpg = cv2.imencode(".jpg", annotated)

if ok:
    display(Image(data=jpg.tobytes()))

---
# Session 8 — Can We Trust Our AI?

## Objective
Test the model under different conditions instead of accepting one successful example.

## New core idea
A model should be **tested**, not just demonstrated.

### Suggested test table

| Test | Expected | Prediction | Confidence | Correct? |
|---|---|---|---:|---|
| Tiger near | tiger |  |  |  |
| Tiger far | tiger |  |  |  |
| Tiger from side | tiger |  |  |  |
| Tiger in dim light | tiger |  |  |  |
| Tiger partly hidden | tiger |  |  |  |
| Empty scene | nothing |  |  |  |
| Different toy animal | nothing |  |  |  |

If the model fails, ask:

> Do we need to change the Python code, or do we need better training examples?

In [ ]:
def inspect_tiger_prediction(image_path, model, confidence_threshold=0.5):
    results = model.predict(
        image_path,
        conf=confidence_threshold,
        verbose=False
    )

    result = results[0]

    if len(result.boxes) == 0:
        print("No tiger detected.")
        return

    for box in result.boxes:
        confidence = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        print(
            f"tiger | confidence={confidence:.2f} | "
            f"box=({x1},{y1}) to ({x2},{y2})"
        )


# Example:
# inspect_tiger_prediction("test_tiger.jpg", tiger_model)

## False-positive test

Show the model:
- a bear
- a cat toy
- an empty floor
- a striped bag
- another orange object

A good project model should not shout **"tiger!"** every time it sees stripes or orange colour.

---
# Session 9 — YOLO Meets UGOT

## Objective
Use the UGOT camera as YOLO's input and turn tiger position into robot movement.

## New core idea

```text
UGOT camera
     ↓
YOLO
     ↓
tiger centre_x
     ↓
LEFT / CENTER / RIGHT
     ↓
UGOT movement
```

### Safety first

During first tests:
1. Put UGOT on the floor with plenty of clear space.
2. Use low movement speeds.
3. Keep one hand ready to lift or stop the robot.
4. Test the **decision-only** version before enabling motors.

## Connect to UGOT

Update the IP address to match your robot.

If your existing teaching environment already creates `got`, you may skip this cell.

In [ ]:
# Run on the computer that can connect to UGOT.

from ugot import ugot

got = ugot.UGOT()

UGOT_IP = "192.168.1.180"   # <-- Change to your UGOT IP
got.initialize(UGOT_IP)

print("UGOT initialized.")

## Read one image from UGOT

The UGOT camera sends compressed image bytes.

We convert:

```text
bytes
  ↓ np.frombuffer
1D NumPy array
  ↓ cv2.imdecode
BGR image
```

In [ ]:
got.open_camera()
print("UGOT camera is on.")

picture_raw = got.read_camera_data()

if picture_raw is None:
    print("No camera data received.")

else:
    picture_1d = np.frombuffer(
        picture_raw,
        np.uint8
    )

    picture_bgr = cv2.imdecode(
        picture_1d,
        cv2.IMREAD_COLOR
    )

    if picture_bgr is None:
        print("Camera data could not be decoded.")

    else:
        ok, jpg = cv2.imencode(
            ".jpg",
            picture_bgr
        )

        if ok:
            display(Image(data=jpg.tobytes()))

## Helper function: get one UGOT camera frame

The function returns:
- a valid OpenCV image, or
- `None` if the camera data is missing

This is useful because a real robot camera may occasionally fail to deliver a frame.

In [ ]:
def get_ugot_frame():
    picture_raw = got.read_camera_data()

    if picture_raw is None:
        return None

    picture_1d = np.frombuffer(
        picture_raw,
        np.uint8
    )

    frame = cv2.imdecode(
        picture_1d,
        cv2.IMREAD_COLOR
    )

    return frame

## Helper function: find the best tiger

There could be more than one detection.

For this project, we choose the tiger detection with the **highest confidence**.

In [ ]:
def find_best_tiger(frame, model, conf=0.60):
    results = model.predict(
        frame,
        conf=conf,
        verbose=False
    )

    result = results[0]

    if len(result.boxes) == 0:
        return None, result

    best_box = max(
        result.boxes,
        key=lambda box: float(box.conf[0])
    )

    x1, y1, x2, y2 = map(
        int,
        best_box.xyxy[0]
    )

    confidence = float(best_box.conf[0])

    tiger = {
        "x1": x1,
        "y1": y1,
        "x2": x2,
        "y2": y2,
        "center_x": (x1 + x2) // 2,
        "center_y": (y1 + y2) // 2,
        "width": x2 - x1,
        "height": y2 - y1,
        "confidence": confidence
    }

    return tiger, result

## Decision-only test

Before moving UGOT, check that the program can correctly decide:

```text
LEFT
RIGHT
FORWARD
NO TIGER
```

In [ ]:
def tiger_action(tiger, frame_width):
    if tiger is None:
        return "NO TIGER"

    left_boundary = int(frame_width * 0.40)
    right_boundary = int(frame_width * 0.60)

    if tiger["center_x"] < left_boundary:
        return "LEFT"

    if tiger["center_x"] > right_boundary:
        return "RIGHT"

    return "FORWARD"

In [ ]:
# Decision-only live test.
# This cell does NOT move the robot.

display_area = None
start_time = time.time()

while time.time() - start_time < 10:

    frame = get_ugot_frame()

    if frame is None:
        print("No camera frame.")
        time.sleep(0.1)
        continue

    tiger, result = find_best_tiger(
        frame,
        tiger_model,
        conf=0.60
    )

    action = tiger_action(
        tiger,
        frame.shape[1]
    )

    print("Decision:", action)

    annotated = result.plot()

    ok, jpg = cv2.imencode(
        ".jpg",
        annotated
    )

    if ok:
        image = Image(data=jpg.tobytes())

        if display_area is None:
            display_area = display(
                image,
                display_id=True
            )
        else:
            display_area.update(image)

    time.sleep(0.1)

## Now allow UGOT to align with the tiger

Use small speeds first.

For a mecanum vehicle:

```text
negative x → shift left
positive x → shift right
positive y → move forward
```

The exact direction can depend on your UGOT configuration.  
Test left/right at low speed before using the full program.

In [ ]:
def move_toward_tiger(action):
    if action == "LEFT":
        got.mecanum_move_xyz(-8, 0, 0)

    elif action == "RIGHT":
        got.mecanum_move_xyz(8, 0, 0)

    elif action == "FORWARD":
        got.mecanum_move_xyz(0, 10, 0)

    else:
        got.mecanum_stop()

---
# Session 10 — Final Project: Find → Align → Approach → Stop

## Objective
Combine YOLO, Python decisions, UGOT movement, and the laser sensor.

## New core idea: Sensor Fusion

YOLO is good at:

```text
WHAT is it?
WHERE is it?
```

The laser sensor is good at:

```text
HOW CLOSE is it?
```

So we combine them:

```text
Camera → YOLO → tiger position
                   ↓
              Python decision
                   ↓
              UGOT movement
                   ↓
Laser sensor → safe stop distance
```

This is more reliable than estimating distance only from bounding-box size.

## Step 1 — Test the laser sensor

For the example below we use sensor ID `21`, which is used in the existing UGOT teaching code.

Check that this ID matches your actual robot setup.

In [ ]:
distance = got.read_distance_data(21)

print(
    f"Laser distance = {distance}"
)

## Step 2 — Final autonomous program

### Behaviour

1. If no tiger is detected → stop.
2. If tiger is left → shift left.
3. If tiger is right → shift right.
4. If tiger is centred and still far → move forward.
5. If laser says tiger is close enough → stop and finish.

### Why stop when the tiger disappears?

For a classroom robot, the safest default is:

> **No reliable visual target = no movement.**

An optional search behaviour can be added later.

In [ ]:
# ============================================================
# FINAL PROJECT
# UGOT finds a toy tiger, aligns, approaches, and stops safely.
# ============================================================

TIGER_CONFIDENCE = 0.60

LEFT_ZONE = 0.40
RIGHT_ZONE = 0.60

SIDE_SPEED = 8
FORWARD_SPEED = 10

LASER_ID = 21
STOP_DISTANCE = 20   # Adjust after measuring your real setup.

display_area = None


try:
    got.open_camera()
    print("Tiger Hunter started.")

    while True:

        # ----------------------------------------------------
        # SENSE 1: CAMERA
        # ----------------------------------------------------
        frame = get_ugot_frame()

        if frame is None:
            got.mecanum_stop()
            print("Camera data missing -> STOP")
            time.sleep(0.1)
            continue


        # ----------------------------------------------------
        # THINK 1: YOLO
        # ----------------------------------------------------
        tiger, result = find_best_tiger(
            frame,
            tiger_model,
            conf=TIGER_CONFIDENCE
        )


        if tiger is None:
            action = "NO TIGER"
            got.mecanum_stop()


        else:
            frame_width = frame.shape[1]

            left_boundary = int(
                frame_width * LEFT_ZONE
            )

            right_boundary = int(
                frame_width * RIGHT_ZONE
            )


            # ------------------------------------------------
            # SENSE 2: LASER
            # ------------------------------------------------
            laser_distance = got.read_distance_data(
                LASER_ID
            )


            # ------------------------------------------------
            # THINK 2: DECISION
            # ------------------------------------------------

            # A positive reading is treated as valid.
            if (
                laser_distance > 0
                and laser_distance <= STOP_DISTANCE
            ):
                action = "TARGET REACHED"
                got.mecanum_stop()

                print(
                    f"Tiger reached! "
                    f"Distance = {laser_distance}"
                )

                break


            elif tiger["center_x"] < left_boundary:
                action = "MOVE LEFT"

                got.mecanum_move_xyz(
                    -SIDE_SPEED,
                    0,
                    0
                )


            elif tiger["center_x"] > right_boundary:
                action = "MOVE RIGHT"

                got.mecanum_move_xyz(
                    SIDE_SPEED,
                    0,
                    0
                )


            else:
                action = "MOVE FORWARD"

                got.mecanum_move_xyz(
                    0,
                    FORWARD_SPEED,
                    0
                )


        # ----------------------------------------------------
        # DISPLAY INFORMATION
        # ----------------------------------------------------
        annotated = result.plot()

        cv2.putText(
            annotated,
            action,
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        # Draw LEFT/CENTER/RIGHT zone boundaries.
        frame_width = annotated.shape[1]

        left_line = int(
            frame_width * LEFT_ZONE
        )

        right_line = int(
            frame_width * RIGHT_ZONE
        )

        cv2.line(
            annotated,
            (left_line, 0),
            (left_line, annotated.shape[0]),
            (100, 100, 100),
            2
        )

        cv2.line(
            annotated,
            (right_line, 0),
            (right_line, annotated.shape[0]),
            (100, 100, 100),
            2
        )

        ok, jpg = cv2.imencode(
            ".jpg",
            annotated
        )

        if ok:
            image = Image(data=jpg.tobytes())

            if display_area is None:
                display_area = display(
                    image,
                    display_id=True
                )
            else:
                display_area.update(image)


        time.sleep(0.1)


except KeyboardInterrupt:
    print("Program stopped by user.")


finally:
    # IMPORTANT:
    # Always stop the motors when the program finishes or crashes.
    got.mecanum_stop()
    print("UGOT stopped safely.")

# Final Student Challenge

## Mission

Place the toy tiger somewhere in front of UGOT.

The robot must:

1. Recognise the tiger.
2. Decide whether it is left, centre, or right.
3. Align itself.
4. Move toward the tiger.
5. Stop at the chosen safe distance.

---

## Success Criteria

### Level 1
YOLO correctly detects the toy tiger.

### Level 2
The robot moves left/right to centre the tiger.

### Level 3
The robot approaches the tiger.

### Level 4
The laser sensor stops the robot safely.

### Level 5
The program works under at least three different:
- tiger positions
- backgrounds
- distances

---

## Student Reflection

1. What information did YOLO provide?
2. What information did the laser sensor provide?
3. Which part of the program made the decision?
4. When did your model fail?
5. What new training images could improve it?

# Optional Extensions

These should come **after** the core project works reliably.

## Extension A — Search for the tiger

Instead of stopping when no tiger is visible:

```python
got.mecanum_move_xyz(0, 0, 8)
```

could slowly rotate the robot.

Add a time limit so it does not rotate forever.

## Extension B — Multiple animals

Train:

```text
tiger
bear
elephant
```

Then give UGOT a mission:

```text
Find only the tiger.
```

## Extension C — Confidence-based behaviour

Example:

```text
confidence < 0.50  → stop
0.50–0.70          → move slowly
> 0.70              → normal movement
```

## Extension D — Compare KNN and YOLO

Ask students:

- What information did KNN give us?
- What extra information does YOLO give us?
- Why is object detection useful for robotics?

This makes YOLO a natural next step after simpler image-classification projects.

# Teacher Notes — Recommended Teaching Order

Do **not** begin by explaining:
- convolution
- backpropagation
- loss functions
- feature maps
- anchors
- non-max suppression mathematics

For this age group, first build the functional mental model:

```text
picture
  ↓
YOLO
  ↓
class + confidence + box
  ↓
Python logic
  ↓
robot action
```

Once students can predict the robot's behaviour before running the code, they have a strong foundation for learning deeper machine-learning concepts later.

---

## Classroom Safety

- Use low motor speeds during debugging.
- Keep a clear operating area.
- Stop the robot when camera data is missing.
- Stop the robot when no reliable tiger detection exists.
- Keep `finally: got.mecanum_stop()` in real-robot programs.
- Calibrate `STOP_DISTANCE` with the real toy and real laser position before student testing.